# Domphy:  Domain Phylogeny
This notebook was designed to download genomes by NCBI accession number, search for proteíns with an specified PFAM [or other InterPro HMM profiles], extract those proteins, extract the regions containing the domain and make a phylogenetic tree.

## Install dependencies

In [1]:
#@markdown ##Install programs
#@markdown ### HMMER, MAFFT, TrimAl, Iqtree3
#@markdown ## <br> Python libraries','#@markdown ### Biopython, numpy, pandas, requests','','import os,sys,json,glob,shutil,re','import multiprocessing, subprocess','','try:','	import pandas as pd','except:','	%pip install pandas','	import pandas as pd','','try:','	import numpy as np','except:','	%pip install numpy
import os,sys,json,glob,shutil,re,gzip
import multiprocessing, subprocess
import pandas as pd
import numpy as np

try:
	import requests
except:
	%pip install requests
	import requests

try:
	from Bio import SeqIO
	from Bio.Seq import Seq
	from Bio.SeqRecord import SeqRecord
except:
	print('Installing Biopython')
	%pip install biopython
	from Bio import SeqIO
	from Bio.Seq import Seq
	from Bio.SeqRecord import SeqRecord

#NCBI datasets CLI
if not os.path.exists('./datasets'):
	print('Installing Datasets')
	!wget https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets 2> /dev/null 1>> logs.txt
	!chmod +x datasets
else:
	print('Datasets already installed...')

try:
	from tqdm import tqdm
except:
	%pip install tqdm
	from tqdm import tqdm

hmmsearch = 'hmmsearch'
if not shutil.which(hmmsearch):
		os.system('sudo apt install hmmer')
		print(f'HMMER is installed: {shutil.which(hmmsearch)}')
else:
		print(f'HMMER is installed: {shutil.which(hmmsearch)}')

#MAFFT for sequence alignment
mafft = 'mafft'
if not shutil.which(mafft):
	print('installing mafft...')
	!sudo apt-get install mafft > logs.txt 2>&1
	if not shutil.which(mafft) and not shutil.which('mafft.bat'):
		print('MAFFT not found, downloading from official website...')
		!wget https://mafft.cbrc.jp/alignment/software/mafft-7.526-linux.tgz 2> /dev/null 1>> logs.txt
		!tar -xzf mafft-7.526-linux.tgz > logs.txt 2>&1
		os.environ['PATH'] = os.environ['PATH']+ os.pathsep + os.path.join(os.getcwd(), 'mafft-linux64/')
		!rm -rf mafft-7.526-linux*
		if not shutil.which(mafft):
			mafft = 'mafft.bat'
			if not shutil.which(mafft):
				print('MAFFT not found, please install it manually.')
				raise
		print(f'MAFFT installed successfully. cli: {mafft}')
	elif shutil.which(mafft):
		!{mafft} --version
		print(f'MAFFT installed successfully. cli: {mafft}')
	else:
		mafft = 'mafft.bat'
		!{mafft} --version
		print(f'MAFFT installed successfully. cli: {mafft}')
else:
	if shutil.which(mafft):
		!mafft --version
		print(f'MAFFT installed successfully. cli: {mafft}')
	elif shutil.which('mafft.bat'):
		mafft = 'mafft.bat'
		!mafft --version
		print(f'MAFFT installed successfully. cli: {mafft}')
	else:
		print('MAFFT not found, please install it manually.')
		raise

trimal = './trimal'
if not shutil.which(trimal):
	print('installing trimAl...')
	try:
		asset_name = 'trimAl_Linux_x86-64.zip'
		url = 'https://github.com/inab/trimal/releases/latest/download/trimAl_Linux_x86-64.zip'
		print(f"Downloading timAl from: {url}")
		with requests.Session() as session:
			download_response = session.get(url, stream=True)
			download_response.raise_for_status()
			# Save the file
			with open(asset_name, 'wb') as f:
					for chunk in download_response.iter_content(chunk_size=8192):
							f.write(chunk)
			print(f"Successfully downloaded {asset_name}")
			print(f"File saved as: {os.path.abspath(asset_name)}")
		#extract
		print('Extracting trimal...')
		!unzip -o -j {asset_name} -d ./
		os.remove(asset_name)
		if not shutil.which(trimal):
			print('trimal not found, please install it manually.')
			raise
	except:
		print(f"Could not find asset '{asset_name}' in the latest release assets.")
else:
	print(f'trimal installed successfully. cli: {trimal}')

#Iqtree2

iqtree = '/content/iqtree-3.0.1-Linux-intel/bin/iqtree3'
if not shutil.which(iqtree):
	print('installing Iqtree3...')
	try:
		asset_name = 'iqtree-3.0.1-Linux-intel.tar.gz'
		url = 'https://github.com/iqtree/iqtree3/releases/download/v3.0.1/iqtree-3.0.1-Linux-intel.tar.gz'
		print(f"Downloading timAl from: {url}")
		with requests.Session() as session:
			download_response = session.get(url, stream=True)
			download_response.raise_for_status()
			# Save the file
			with open(asset_name, 'wb') as f:
					for chunk in download_response.iter_content(chunk_size=8192):
							f.write(chunk)
			print(f"Successfully downloaded {asset_name}")
			print(f"File saved as: {os.path.abspath(asset_name)}")
		#extract
		print('Extracting Iqtree3...')
		!tar -xzf {asset_name} -C ./
		os.remove(asset_name)
		if not shutil.which(iqtree):
			print('Iqtree3 not found, please install it manually.')
			raise
	except:
		print(f"Could not find asset '{asset_name}' in the latest release assets.")
else:
	print(f'Iqtree3 installed successfully. cli: {iqtree}')


Installing Biopython
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.7 MB/s eta 0:00:00
Installing Datasets
HMMER is installed: /usr/bin/hmmsearch
installing mafft...
v7.490 (2021/Oct/30)
MAFFT installed successfully. cli: mafft
installing trimAl...
Successfully downloaded trimAl_Linux_x86-64.zip
File saved as: /content/trimAl_Linux_x86-64.zip
Extracting trimal...
Archive:  trimAl_Linux_x86-64.zip
  inflating: ./trimal                
  inflating: ./readal                
  inflating: ./statal                
  inflating: ./LICENSE               
installing Iqtree3...
Successfully downloaded iqtree-3.0.1-Linux-intel.tar.gz
File saved as: /content/iqtree-3.0.1-Linux-intel.tar.gz
Extracting Iqtree3...


## Look for proteins containing a domain of interest
Use hmmsearch to look for proteins containing a domain of interest and generate multifasta file with the sequences.
If --extract_domain is used, the sequence of the domain of interest will also be extracted.

In [3]:
#@markdown Input a list of comma separated NCBI assembly accession numbers to download
accs = 'GCF_000006965.1,GCF_000007025.1,GCF_000007505.1,GCF_000010525.1,GCF_000013085.1,GCF_000021865.1,GCF_000022005.1,GCF_000023785.1,GCF_000092045.1,GCF_000967305.2,GCF_001007935.1,GCF_001040945.1,GCF_001458195.1,GCF_001547995.1,GCF_001642675.1,GCF_001718895.1,GCF_001719165.1,GCF_003148495.1,GCF_003324715.1,GCF_004063735.1,GCF_013181415.1,GCF_013318015.2,GCF_014171495.1,GCF_015326295.1,GCF_016466415.2,GCF_016584445.1,GCF_016864595.1,GCF_017894385.1,GCF_019297835.1,GCF_022869645.1,GCF_030761375.1.gb' #@param {type:'string'}
#@markdown <br>
genomes_folder = 'genomes' #@param {type:'string'}
custom_genomes_folder = '' #@param {type:'string'}
results_folder = 'results' #@param {type:'string'}

if not os.path.exists(genomes_folder):
	os.mkdir(genomes_folder)
if not os.path.exists(results_folder):
	os.mkdir(results_folder)

############

def extract_proteins(gf,pf):
	with open(pf,'w') as f:
		for r in SeqIO.parse(gf, "gb"):
			for feature in r.features:
				if feature.type == "CDS":
					if 'locus_tag' in feature.qualifiers:
						name = feature.qualifiers['locus_tag'][0]
					elif 'gene' in feature.qualifiers:
						name = feature.qualifiers['gene'][0]
					elif 'protein_id' in feature.qualifiers:
						name = feature.qualifiers['protein_id'][0]
					else:
						continue
					if 'translation' in feature.qualifiers:
						aaseq = feature.qualifiers['translation'][0]
						f.write(f'>{gf}|{name}\n{aaseq}\n')
	return(pf)

############


genome_files = []
proteome_files = []
if custom_genomes_folder == '' and len(accs) > 0:
	print(f'Downloading genomes:')
	for i,acc in enumerate(accs.replace(' ' ,'').split(',')):
		if (i+1)%5 == 0:
			print()
		acc = acc.strip()
		gbacc = acc
		acc = acc.replace('GCA_','GCF_') # use RefSeq
		gf = f'./{genomes_folder}/{acc}.gb'
		pf = f'./{genomes_folder}/{acc}.faa'
		gf_gb = f'./{genomes_folder}/{gbacc}.gb'
		pf_gb = f'./{genomes_folder}/{gbacc}.faa'
		if not os.path.exists(gf) and not os.path.exists(gf_gb):
			print(f'{acc}', end='  ')
			try:
				!./datasets download genome accession {acc} --include gbff --filename {acc}.zip >/dev/null 2>&1
				!unzip -o -j {acc}.zip ncbi_dataset/data/{acc}/genomic.gbff -d ./ >/dev/null 2>&1
				os.remove(f'{acc}.zip')
				!mv genomic.gbff {genomes_folder}/{acc}.gb >/dev/null 2>&1
				genome_files.append(gf)
				proteome_files.append(extract_proteins(gf,pf))
			except:
				print(f'Error: {acc} not found', end='  ')
				try:
					print(f'{gbacc}', end='  ')
					!./datasets download genome accession {gbacc} --include gbff --filename {gbacc}.zip >/dev/null 2>&1
					!unzip -o -j {gbacc}.zip ncbi_dataset/data/{gbacc}/genomic.gbff -d ./ >/dev/null 2>&1
					os.remove(f'{gbacc}.zip')
					!mv genomic.gbff {genomes_folder}/{gbacc}.gb >/dev/null 2>&1
					genome_files.append(gf_gb)
					proteome_files.append(extract_proteins(gf_gb,pf_gb))
					print(f'{gf_gb} successfully downloaded', end='  ')
				except:
					print(f'Error: {gbacc} not found', end='  ')
		else:
			if os.path.exists(gf):
				print(f'{gf} already downloaded.', end='  ')
				genome_files.append(gf)
				if not os.path.exists(pf):
					print(f'Extracting proteome from {gf}', end='  ')
					proteome_files.append(extract_proteins(gf,pf))
				else:
					proteome_files.append(pf)
					print(f'{pf} already extracted.', end='  ')
			elif os.path.exists(gf_gb):
				print(f'{gf_gb} already downloaded.', end='  ')
				genome_files.append(gf_gb)
				if not os.path.exists(pf_gb):
					proteome_files.append(extract_proteins(gf_gb,pf_gb))
				else:
					proteome_files.append(pf_gb)
					print(f'{pf_gb} already extracted.', end='  ')
			else:
				print(f'Neither {gf} nor {gf_gb} were found...')
elif custom_genomes_folder != '':
	if not os.path.exists(custom_genomes_folder):
		print(f'ERROR: {custom_genomes_folder} not found.')
	else:
		genome_files = [os.path.join(custom_genomes_folder,f) for f in os.listdir(custom_genomes_folder) if f.endswith('.gb')]
		print(f'Custom dataset selected. {len(genome_files)} genome files found in {genomes_folder} folder.')
		print(f'Extracting proteomes:')
		for i,gf in enumerate(genome_files):
			if (i+1)%5 == 0:
				print()
			pf = f'{genomes_folder}/{os.path.splitext(os.path.basename(gf))[0]}.faa'
			if not os.path.exists(pf):
				proteome_files.append(extract_proteins(gf,pf))
				print(f'{gf}', end='  ')
else:
	print('No genomes specified.')

ngenomes = len(genome_files)
print(f'\n{ngenomes} genomes successfully downloaded')

GCF_000006965.1  GCF_000007025.1  GCF_000007505.1  GCF_000010525.1  
GCF_000013085.1  GCF_000021865.1  GCF_000022005.1  GCF_000023785.1  GCF_000092045.1  
GCF_000967305.2  GCF_001007935.1  GCF_001040945.1  GCF_001458195.1  GCF_001547995.1  
GCF_001642675.1  GCF_001718895.1  GCF_001719165.1  GCF_003148495.1  GCF_003324715.1  
GCF_004063735.1  GCF_013181415.1  GCF_013318015.2  GCF_014171495.1  GCF_015326295.1  
GCF_016466415.2  GCF_016584445.1  GCF_016864595.1  GCF_017894385.1  GCF_019297835.1  
GCF_022869645.1  GCF_030761375.1.gb  Error: GCF_030761375.1.gb not found  GCF_030761375.1.gb  Error: GCF_030761375.1.gb not found  
30 genomes successfully downloaded


### Download PFAM and other HMM profiles from Interpro
Pleas input the PFAM, PANTHER, CATH, SUPERFAMILY OR TIGRFam IDS to download the HMM profiles.


In [4]:
domain_id = 'PF00072' #@param {type: "string"}
domain_hmm_file = os.path.join(results_folder,f'{domain_id}.hmm')

def download_pfam_hmm(pfamid, pfam_file):
		success = False
		if (os.path.exists(pfam_file)) and (os.path.getsize(pfam_file) > 1024):
				print(f'--> {pfamid} already downloaded...')
				return
		print(f'--> Downloading {pfamid}...')
		urls = {'PFAM':f'https://www.ebi.ac.uk/interpro/wwwapi//entry/pfam/{pfamid}?annotation=hmm',
				'PANTHER':f'https://www.ebi.ac.uk/interpro/wwwapi//entry/panther/{pfamid}?annotation=hmm',
				'CATH':f'https://www.ebi.ac.uk/interpro/wwwapi//entry/cathgene3d/{pfamid}?annotation=hmm',
				'SUPERFAMILY': f'https://www.ebi.ac.uk/interpro/wwwapi//entry/ssf/{pfamid}?annotation=hmm',
				'TIGRFam':f'https://www.ebi.ac.uk/interpro/wwwapi//entry/ncbifam/{pfamid}?annotation=hmm'}
		for url in urls:
			try:
				r = requests.get(urls[url], headers={'Accept-Encoding': 'gzip'}, stream=True)
				if not r.ok:
					continue
				else:
					hmm = str(gzip.decompress(r.content), 'utf-8')
					with open(pfam_file,'w+') as hf:
						hf.write(hmm)
					success = True
			except:
				success = False
		if not success:
				print(f'\nERROR: {pfamid} couldn\'t be downloaded.')
		else:
			print(f'\n{pfamid} downloaded successfully.')
		return

download_pfam_hmm(domain_id, domain_hmm_file)


--> Downloading PF00072...

PF00072 downloaded successfully.


### Use only if you have a custom HMM or have already downloaded it

In [ ]:
domain_hmm_file = '' #@param {type: "string"}
if not os.path.exists(domain_hmm_file):
	print(f"ERROR: {domain_hmm_file} not found.")
else:
	print(f"{domain_hmm_file} found.")


histidine_kinase.hmm found.


### Search for the specified domain using hmmsearch
Please input the path to the proteome files, the domain hmm file, the results folder, the e-value and the hmm coverage.
E-value: evalue <number>
HMM coverage: hmm_coverage (percentage) <number>

In [5]:
#@markdown ### HMMsearch options
#@markdown <br> E-value and hmm coverage (percent)
evalue = 1e-5 #@param {type:'number'}
hmm_coverage = 70 #@param {type:'number'}


#@markdown ### Proteome fasta (used if no genomes are provided). A file with all the target proteomes in fasta format.
proteome_fasta = 'all_homologs_sequences.faa' #@param {type:'string'}

if ngenomes == 0 and len(proteome_files) == 0:
	print('No genomes to process. Using proteome fasta.')
	if not os.path.exists(proteome_fasta):
		print(f'ERROR: {proteome_fasta} not found.')
	else:
		print(f'Using proteome fasta: {proteome_fasta}')
		pfs = [proteome_fasta]
		use_proteome_fasta = True
else:
	use_proteome_fasta = False
	pfs = proteome_files


def hmmsearch(proteomeFile,hmm, genome_folder, evalue, hmm_coverage):
	hmm_name = os.path.splitext(os.path.basename(hmm))[0]
	resFile = os.path.join(genome_folder, hmm_name + ".txt")
	subprocess.call(['hmmsearch', "--domtblout", resFile , hmm , proteomeFile], stdout=subprocess.DEVNULL)
	tableHeaders = ['target name', 'accession', 'tlen', 'query name', 'qaccession', 'qlen', 'E-value', 'score', 'bias', '#', 'of', 'c-Evalue', 'i-Evalue', 'dom score', 'dom bias', 'hmm from', 'hmm to', 'ali from', 'ali to', 'env from', 'env to', 'acc', 'description of target']
	try:
		hmmHits = pd.read_table(resFile, comment='#', sep='\\s+', header=None, usecols=range(22))
		os.remove(resFile)
	except:
		return(None,None)
	if len(hmmHits) > 0:
		renameDict = { old:new for old,new in zip(hmmHits.columns[0:len(tableHeaders)],tableHeaders)}
		hmmHits = hmmHits.rename(columns=renameDict)
	hmmHits = hmmHits[hmmHits['i-Evalue'] < evalue]
	#coverage
	hmmHits['hmm_coverage'] = (hmmHits['hmm to'] - hmmHits['hmm from'])/(hmmHits['qlen'])*100
	hmmHits = hmmHits[hmmHits['hmm_coverage'] > hmm_coverage]
	proteins = list(hmmHits['target name'].unique())
	return(hmmHits,proteins)

tableHeaders = ['target name', 'accession', 'tlen', 'query name', 'qaccession', 'qlen', 'E-value', 'score', 'bias', '#', 'of', 'c-Evalue', 'i-Evalue', 'dom score', 'dom bias', 'hmm from', 'hmm to', 'ali from', 'ali to', 'env from', 'env to', 'acc', 'description of target']
all_hits_table = pd.DataFrame(columns = tableHeaders)
genomes_without_domains = []
for pf in tqdm(pfs):
	hmmhits, proteins = hmmsearch(pf, domain_hmm_file, results_folder, evalue, hmm_coverage)
	if len(all_hits_table) == 0 and hmmhits is not None:
		all_hits_table = hmmhits
	elif hmmhits is not None:
		all_hits_table = pd.concat([all_hits_table, hmmhits])
	else:
		genomes_without_domains.append(pf)

all_hits_table.to_csv(os.path.join(results_folder, 'all_hits_table.csv'), index=False)
print(f'Genomes without domains: {" -- ".join(genomes_without_domains)}')
#extract proteins with domains
extracted = []
rs = []
domains = []
for _,row in tqdm(all_hits_table.iterrows()):
	protein = row['target name']
	domain_name = row['query name']
	start = row['env from']
	end	= row['env to']
	if use_proteome_fasta:
		proteome_file = proteome_fasta
	else:
		proteome_file = os.path.splitext(protein.split('|')[0])[0] + '.faa'
	locus = protein.split('|')[1]
	for r in SeqIO.parse(proteome_file, 'fasta'):
		pid = r.id.split('|')[1]
		if pid == locus:
			if protein not in extracted:
				rs.append(r)
				extracted.append(protein)
			new_id = f'{r.id}_{start}-{end}_{domain_name}'
			new_rec = SeqRecord(r.seq[start:end], id=new_id, description='')
			domains.append(new_rec)

SeqIO.write(rs, os.path.join(results_folder, 'proteins_with_domains.fasta'), 'fasta')
SeqIO.write(domains, os.path.join(results_folder, 'domains.fasta'), 'fasta')
print(f'{len(rs)} proteins with domains extracted')
print(f'{len(domains)} domains extracted')

100%|██████████| 30/30 [00:03<00:00,  8.96it/s]


Genomes without domains: 


1783it [00:33, 52.63it/s]

1709 proteins with domains extracted
1783 domains extracted


In [6]:
#@markdown ###Rename Domain files
records = []
def get_only_id(record):
    return record.id

renamed_domains = os.path.join(results_folder, 'domains_rn.fasta')
with open(os.path.join(results_folder, 'domains_rn.fasta'), 'w') as out:
	for r in SeqIO.parse(os.path.join(results_folder, 'domains.fasta'),'fasta'):
		try:
			r.id = f'GC{r.id.split("/GC")[1]}'
			out.write(f'>{r.id}\n{r.seq}\n')
		except:
			print(f'Id format different from the expected: {r.id}')
			renamed_domains = os.path.join(results_folder, 'domains.fasta')
			break


### Mafft alignment
Makes an alignment using mafft. <br> If the number of sequences is less than 1000, the ginsi mode: --maxiterate 1000 --globalpair is used, else mafft is run in auto mode.<br>

<br>Trimal: If you want to clean the alignment with trimAL select clean_with_trimAl. The gappyout mode will be used.

<br>Iqtree3: For phylogenetic tree inference Iqtree3 will be used. If there are more than 1000 sequences the fast mod will be used


In [ ]:
#@markdown ###Make Domain Phylogenetic tree
clean_with_trimAl = True #@param {type:'boolean'}
make_ml_tree = True #@param {type:'boolean'}
trimAl_mode = 'gappyout' #@param ['automated1', 'gappyout','strict','strictplus','custom']
trimAl_mode = '-'+trimAl_mode
#@markdown For custom trimAL arguments sue, for example: '-gt 0.8 -st 0.5' etc..
if trimAl_mode == 'custom':
  custom_args = '' #@param {type:'string'}
  trimAl_mode = custom_args

#@markdown Tree options
bootstrap_replicants = 1000 #@param {type:'integer'}
success = False
seqs_file = renamed_domains
aln_file = os.path.join(results_folder, "domains_aln.fasta")
import subprocess


# Open the output file for writing

try:
  if len(domains) > 1000:
    print('Making alignment...\n{mafft} --auto --thread -1 {seqs_file} > {aln_file}')
    # mafft command
    command_args = [mafft, "--auto", "--reorder", "--thread", "-1", seqs_file ]
    with open(aln_file, "w") as f:
      result = subprocess.run(command_args, stdout=f, stderr=subprocess.PIPE, text=True, check=True)
    success = True
    print(f"Alignment complete. Output saved to {aln_file}")
  else:
    print('Making alignment...\n{mafft} --maxiterate 1000 --globalpair --reorder --thread -1 {seqs_file} > {aln_file}')
    command_args = [mafft, "--maxiterate", "1000", "--globalpair", "--reorder", "--thread", "-1", seqs_file ]
    with open(aln_file, "w") as f:
      result = subprocess.run(command_args, stdout=f, stderr=subprocess.PIPE, text=True, check=True)
    success = True
    print(f"Alignment complete. Output saved to {aln_file}")
except:
  print('Error with alignment')
  success = False

if success:
  try:
    print('Cleaning domain aligment...')
    if clean_with_trimAl:
      os.system(f'{trimal} -in {os.path.join(results_folder, "domains_aln.fasta")} -out {os.path.join(results_folder, "domains_aln_trimmed.fasta")} {trimAl_mode}')
      aln = os.path.join(results_folder, "domains_aln_trimmed.fasta")
    else:
      aln = os.path.join(results_folder, "domains_aln.fasta")
    success = True
  except:
    print('Error with trimming')
    success = False

if success and make_ml_tree:
  try:
    print('Inferring tree...')
    if len(domains) > 1000:
      fast = ' -fast'
      print('Setting branch support to SH-aLRT test (Approximate Likelihood Ratio Test). Ultrafast bootstrap is not compatible with -fast mode...')
      if bootstrap_replicants > 0:
        bst = f' -alrt {bootstrap_replicants}'
      else:
        bst = ''
    else:
      fast = ''
      if bootstrap_replicants > 0:
        bst = f' -B {bootstrap_replicants}'
      else:
        bst = ''

    os.system(f'{iqtree} -s {aln} -T AUTO -m TEST{bst}{fast}')
    success = True
  except:
    print('Error with tree inference')
    success = False

if success:
  print('Job done!')

Making alignment...
{mafft} --auto --thread -1 {seqs_file} > {aln_file}
Alignment complete. Output saved to results/domains_aln.fasta
Cleaning domain aligment...
Inferring tree...
Setting branch support to SH-aLRT test (Approximate Likelihood Ratio Test). Ultrafast bootstrap is not compatible with -fast mode...
